## Tokenizer

In [1]:
with open("TheMetamorphosis.txt", "r", encoding = "utf-8") as f:
    raw_text = f.read()

print("total chars: ", len(raw_text))
print(raw_text[:100])
    

total chars:  118421
One morning, when Gregor Samsa woke from troubled dreams, he found
himself transformed in his bed in


In [2]:
import re

data = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
data = [word.strip() for word in data if word.strip()]

print(data[:39])
print("total words: ", len(data))

['One', 'morning', ',', 'when', 'Gregor', 'Samsa', 'woke', 'from', 'troubled', 'dreams', ',', 'he', 'found', 'himself', 'transformed', 'in', 'his', 'bed', 'into', 'a', 'horrible', 'vermin', '.', 'He', 'lay', 'on', 'his', 'armour-like', 'back', ',', 'and', 'if', 'he', 'lifted', 'his', 'head', 'a', 'little', 'he']
total words:  24331


In [3]:
words = sorted(set(data))
vocab_size = len(words)

print(words[:30])
print(vocab_size)

['!', '(', ')', ',', '.', ':', ';', '?', 'A', 'Across', 'After', 'All', 'Although', 'An', 'And', 'Anna', 'Another', 'Anyway', 'Aren’t', 'Arms', 'As', 'At', 'Be', 'Because', 'Before', 'Behind', 'Besides', 'But', 'By', 'Can’t']
2952


In [4]:
vocab = {s:i for i,s in enumerate(words)}
print(len(vocab.items()))

2952


In [5]:
class Tokenizer:
    def __init__(self, vocab):

        self.stoi = vocab
        self.itos = {i:s for s,i in vocab.items()}

    def encode(self, text):
        data = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        data = [word.strip() for word in data if word.strip()]

        ids = [self.stoi[i] for i in data]
        return ids
    def decode(self, ids):
        text = " ".join(self.itos[i] for i in ids)
        text = re.sub(r's+([,.:;?_!"()\'])', r'\1', text)
        return text

In [6]:
t1 = Tokenizer(vocab)

text = "when Gregor Samsa woke from troubled dreams"
ids = t1.encode(text)
decoded_text = t1.decode(ids)

print("encoded ids:",ids)
print("decoded text:",decoded_text)

encoded ids: [2764, 53, 114, 2810, 1125, 2599, 845]
decoded text: when Gregor Samsa woke from troubled dreams


In [7]:
words.extend(["<|endoftext|>", "<|unk|>"])
vocab_size = len(words)

vocab = {s:i for i,s in enumerate(words)}

In [8]:
class TokenizerV2:
    def __init__(self, vocab):

        self.stoi = vocab
        self.itos = {i:s for s,i in vocab.items()}

    def encode(self, text):
        data = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        data = [word.strip() for word in data if word.strip()]

        data = [word if word in self.stoi else "<|unk|>" for word in data]
        
        ids = [self.stoi[i] for i in data]
        return ids

        
    def decode(self, ids):
        text = " ".join(self.itos[i] for i in ids)
        text = re.sub(r's+([,.:;?_!"()\'])', r'\1', text)
        return text

In [9]:
t2 = TokenizerV2(vocab)

text = "Fucking Gregor Samsa woke from troubled dreams"
ids = t2.encode(text)
decoded_text = t2.decode(ids)

print("encoded ids:",ids)
print("decoded text:",decoded_text)

encoded ids: [2953, 53, 114, 2810, 1125, 2599, 845]
decoded text: <|unk|> Gregor Samsa woke from troubled dreams


In [10]:
#! pip install tiktoken


In [11]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


In [12]:
tokenizer = tiktoken.get_encoding("gpt2")

In [13]:
text = ("Fucking Gregor Samsa woke from troubled dreams")

tIds = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
dText = tokenizer.decode(tIds)

print(tIds)
print(dText)

[37, 19296, 8547, 273, 3409, 11400, 19092, 422, 17840, 10625]
Fucking Gregor Samsa woke from troubled dreams


## DataLoader

In [14]:
#! pip install torch

In [15]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, context_len, stride):
        self.input_ids = []
        self.output_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - context_len, stride):
            input_chunk = token_ids[i: i + context_len]
            output_chunk = token_ids[i + context_len: i + context_len + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.output_ids.append(torch.tensor(output_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.output_ids[idx]
        
        

In [16]:
def GPTDataLoader(text, batch_size = 8, context_len = 4, stride = 4, shuffle = False, drop_last = True, num_workers = 0):

    dataset = GPTDataset(text, tokenizer, context_len, stride)

    dataloader = DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [17]:
dataloader = GPTDataLoader(raw_text, batch_size=8, context_len=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter) 
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[ 3198,  3329,    11,   618],
        [ 8547,   273,  3409, 11400],
        [19092,   422, 17840, 10625],
        [   11,   339,  1043,   198],
        [38400,   944, 14434,   287],
        [  465,  3996,   656,   257],
        [12361,  3326,  1084,    13],
        [  679,  3830,   319,   465]])

Targets:
 tensor([[ 8547],
        [19092],
        [   11],
        [38400],
        [  465],
        [12361],
        [  679],
        [  198]])


## Embeddings

In [18]:
output_dims = 256
vocab_size = 50257

In [19]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[ 3198,  3329,    11,   618],
        [ 8547,   273,  3409, 11400],
        [19092,   422, 17840, 10625],
        [   11,   339,  1043,   198],
        [38400,   944, 14434,   287],
        [  465,  3996,   656,   257],
        [12361,  3326,  1084,    13],
        [  679,  3830,   319,   465]])

Inputs shape:
 torch.Size([8, 4])


In [20]:
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dims)
token_embeddings = token_embedding_layer(inputs)

In [21]:
context_len = 4
pos_embedding_layer = torch.nn.Embedding(context_len, output_dims)

In [22]:
pos_embeddings = pos_embedding_layer(torch.arange(context_len))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [23]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


## Simplified Attention w/o trainable weights

In [24]:
# we will take new input embedding for simplicity with a dimensionality of 3 rather than 256.
inputs = torch.tensor( 
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [25]:
query = inputs[-1] #step as the query

attn_score_Q = torch.empty(inputs.shape[0]) # values inside are random, we take torch.empty just for the example
print(attn_score_Q)

for i, xi in enumerate(inputs):
    attn_score_Q[i] = torch.dot(xi, query)

print(attn_score_Q)

tensor([0., 0., 0., 0., 0., 0.])
tensor([0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450])


In [26]:
attn_weight_Q = torch.softmax(attn_score_Q, dim = 0)
print(attn_weight_Q)

tensor([0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896])


In [27]:
context_vec_Q = torch.zeros(query.shape)

for i,xi in enumerate(inputs):
    context_vec_Q += attn_weight_Q[i] * xi

print(context_vec_Q)
    

tensor([0.4177, 0.6503, 0.5645])


In [28]:
# now we do it for all of the words in the input.

inputs = torch.tensor( 
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)
attn_scores = inputs @ inputs.T
print(attn_scores)


tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [29]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [30]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## Self Attention

In [31]:
import torch.nn as nn
class SelfAttention(nn.Module):

    def __init__(self, dim_in, dim_out, qkv_bias = False):
        super().__init__()
        self.query= nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.key = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.value = nn.Linear(dim_in, dim_out, bias = qkv_bias)

    def forward(self, x):
        queries = self.query(x)
        keys = self.key(x)
        values = self.value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores /  keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values

        return context_vec
        

In [32]:
sa_v2 = SelfAttention(3, 2)
print(sa_v2(inputs))

tensor([[-0.3279,  0.0101],
        [-0.3284,  0.0094],
        [-0.3284,  0.0096],
        [-0.3283,  0.0100],
        [-0.3284,  0.0132],
        [-0.3284,  0.0082]], grad_fn=<MmBackward0>)


## Causal Attention


In [33]:
class CausalAttention(nn.Module):

    def __init__(self, dim_in, dim_out, context_len,  dropout, qkv_bias = False):
        super().__init__()

        self.query = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.key = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.value = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask",torch.triu(torch.ones(context_len, context_len), diagonal = 1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape #new batch dimension -> b
        
        queries = self.query(x)
        keys = self.key(x)
        values = self.value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[1]** 0.5, dim = -1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values

        return context_vec

In [34]:
batch = torch.stack((inputs, inputs), dim=0)

In [35]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(3, 2, 6, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)
context_vecs

context_vecs.shape: torch.Size([2, 6, 2])


tensor([[[-0.4519,  0.2216],
         [-0.5856,  0.0087],
         [-0.6284, -0.0607],
         [-0.5664, -0.0832],
         [-0.5511, -0.0978],
         [-0.5290, -0.1073]],

        [[-0.4519,  0.2216],
         [-0.5856,  0.0087],
         [-0.6284, -0.0607],
         [-0.5664, -0.0832],
         [-0.5511, -0.0978],
         [-0.5290, -0.1073]]], grad_fn=<UnsafeViewBackward0>)

## Multi-head Attention


In [37]:
class MultiHeadAttentionV0(nn.Module):

    def __init__(self, dim_in, dim_out, context_len, dropout, num_heads, qkv_bias = False):
        super().__init__()

        self.heads = nn.ModuleList()
        for _ in range(num_heads):
            head = CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
            self.heads.append(head)

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim = -1)

## multihead Attention with Weight Splits 

In [47]:
class MultiHeadAttention(nn.Module):

    def __init__(self, dim_in, dim_out, context_len, dropout, num_heads, qkv_bias = False):
        super().__init__()

        if d_out % num_heads != 0:
            raise ValueError("d_out must be divisible by num_heads")

        self.dim_out = dim_out
        self.num_heads = num_heads
        self.head_dim = dim_out // num_heads

        self.query = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.key = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.value = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)

        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs

        self.register_buffer("mask",torch.triu(torch.ones(context_len, context_len), diagonal = 1))

    def forward(self, x):
        b, num_tokens, dim_in = x.shape

        queries = self.query(x)
        keys = self.key(x)
        values = self.value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        
        context_vec = (attn_weights @ values).transpose(1, 2) # Shape: (b, num_tokens, num_heads, head_dim)

        context_vec = context_vec.reshape(b, num_tokens, self.dim_out)

        return context_vec
        

In [48]:
torch.manual_seed(123)

# Define the tensor with 3 rows and 6 columns
inputs = torch.tensor(
    [[0.43, 0.15, 0.89, 0.55, 0.87, 0.66],  # Row 1
     [0.57, 0.85, 0.64, 0.22, 0.58, 0.33],  # Row 2
     [0.77, 0.25, 0.10, 0.05, 0.80, 0.55]]  # Row 3
)

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) 

batch_size, context_length, dim_in = batch.shape
dim_out = 6
mha = MultiHeadAttention(dim_in, dim_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

torch.Size([2, 3, 6])
tensor([[[-0.1354,  0.2538,  0.1353,  0.0993,  0.5164,  0.1103],
         [-0.2447,  0.3172, -0.0060,  0.1318,  0.6144,  0.0032],
         [-0.2898,  0.2501,  0.0070,  0.0092,  0.5663,  0.0665]],

        [[-0.1354,  0.2538,  0.1353,  0.0993,  0.5164,  0.1103],
         [-0.2447,  0.3172, -0.0060,  0.1318,  0.6144,  0.0032],
         [-0.2898,  0.2501,  0.0070,  0.0092,  0.5663,  0.0665]]],
       grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 3, 6])
